# Análisis complementarios

Este notebook reúne los cálculos que complementan a los notebooks 04 y 05 y que se usan en el informe.
No reajusta ningún modelo: parte de las predicciones de 2019 guardadas en `results/test_predictions.csv`.

1. **Bootstrap adicional por bloques semanales.** Diferencia de NMAE macro entre el Random Forest cíclico y cada comparador
   (referencia semanal, Ridge, árbol podado y Random Forest con dummies), con las mismas réplicas para todos.
2. **Sensibilidad del bootstrap al largo del bloque** (1, 3, 7, 14 y 28 días).

Los intervalos son condicionales al ajuste realizado: no incluyen la incertidumbre de entrenar ni de seleccionar hiperparámetros.

In [ ]:
# Cronometro por seccion y backend (CPU/GPU). Debe ir antes de importar scikit-learn.
import sys
from pathlib import Path

_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(_ROOT / "src"))

import timing

BACKEND = timing.activar_backend()   # EYP_BACKEND=cpu (por defecto) o gpu
CRONO = timing.Cronometro("06_complementary_analyses")
print("Backend:", BACKEND)
CRONO.seccion("Preparacion")

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from modeling import load_data, make_final_split, TARGET, COUNTRIES, RESULTS_DIR

df = load_data()
final_train, _ = make_final_split(df)
mu = final_train.groupby("country")[TARGET].mean()[COUNTRIES].to_numpy()   # escala del NMAE en 2019

pred = pd.read_csv(RESULTS_DIR / "test_predictions.csv", parse_dates=["utc_timestamp"])
start = pd.Timestamp("2019-01-01", tz="UTC")
print("Filas de 2019:", len(pred), "| escala de entrenamiento (MW):", dict(zip(COUNTRIES, mu.round(1))))

Filas de 2019: 26031 | escala de entrenamiento (MW): {'DE': np.float64(55843.8), 'ES': np.float64(28710.4), 'FR': np.float64(53960.6)}


## Funciones

El error absoluto se suma por bloque y país, y el NMAE de cada réplica se calcula como cociente de sumas
(error total / filas totales / media de entrenamiento). Todas las comparaciones usan las mismas listas de bloques remuestreados.

In [ ]:
CRONO.seccion('Funciones')

In [2]:
PRED_COL = {"Estacional semanal": "pred_weekly", "Ridge": "pred_ridge",
            "Árbol podado": "pred_tree", "RF dummies": "pred_rf_dummy", "RF cíclico": "pred_rf"}


def prepare(block_days):
    block = ((pred["utc_timestamp"] - start).dt.days // block_days).to_numpy()
    nb = int(block.max()) + 1
    country = pred["country"].to_numpy()
    n = np.zeros((nb, 3))
    ae = {k: np.zeros((nb, 3)) for k in PRED_COL}
    for j, c in enumerate(COUNTRIES):
        m = country == c
        n[:, j] = np.bincount(block[m], minlength=nb)
        for k, col in PRED_COL.items():
            err = (pred.loc[m, TARGET] - pred.loc[m, col]).abs().to_numpy()
            ae[k][:, j] = np.bincount(block[m], weights=err, minlength=nb)
    return nb, n, ae


def stats(a, b, idx, n, ae):
    cnt = n[idx].sum(0)
    na, nb_ = ae[a][idx].sum(0) / cnt / mu, ae[b][idx].sum(0) / cnt / mu
    # [diferencia macro, diferencia por pais (3), mejora relativa macro, mejora relativa por pais (3)]
    return np.r_[na.mean() - nb_.mean(), na - nb_, 1 - na.mean() / nb_.mean(), 1 - na / nb_]


def run_bootstrap(block_days, comparators, B=2000, seed=42):
    nb, n, ae = prepare(block_days)
    rng = np.random.default_rng(seed)
    draws = [rng.integers(0, nb, nb) for _ in range(B)]       # mismas réplicas para todos los comparadores
    rows = []
    for comp in comparators:
        obs = stats("RF cíclico", comp, np.arange(nb), n, ae)
        boot = np.array([stats("RF cíclico", comp, d, n, ae) for d in draws])
        lo, hi = np.percentile(boot, 2.5, axis=0), np.percentile(boot, 97.5, axis=0)
        labels = ["Macro", *COUNTRIES]
        for i, lab in enumerate(labels):
            rows.append({"bloque_dias": block_days, "n_bloques": nb, "comparador": comp, "nivel": lab,
                         "delta": obs[i], "delta_inf": lo[i], "delta_sup": hi[i],
                         "P(delta>=0)": (boot[:, i] >= 0).mean(),
                         "mejora": obs[4 + i], "mejora_inf": lo[4 + i], "mejora_sup": hi[4 + i]})
    return pd.DataFrame(rows)

## 1. Bootstrap con bloques de 7 días: todos los comparadores

Primero se verifica que la comparación contra la referencia semanal reproduce lo exportado por el notebook 05 (`results/test_bootstrap_ci.csv`).

In [ ]:
CRONO.seccion('1. Bootstrap con bloques de 7 días: todos los comparadores')

In [3]:
comparators = ["Estacional semanal", "Ridge", "Árbol podado", "RF dummies"]
extra = run_bootstrap(7, comparators)

chk = pd.read_csv(RESULTS_DIR / "test_bootstrap_ci.csv", index_col=0)
mine = extra[(extra.comparador == "Estacional semanal")].set_index("nivel")
assert np.allclose(mine.loc["Macro", "delta"], chk.loc["Δ NMAE macro", "observado"], atol=1e-9)
assert np.allclose(mine.loc["Macro", "delta_inf"], chk.loc["Δ NMAE macro", "IC95 inf"], atol=1e-6)
assert np.allclose(mine.loc["Macro", "delta_sup"], chk.loc["Δ NMAE macro", "IC95 sup"], atol=1e-6)
print("Reproduce el bootstrap del notebook 05.")

extra.to_csv(RESULTS_DIR / "test_bootstrap_extra.csv", index=False)
extra[extra.nivel == "Macro"].drop(columns=["nivel"]).round(4)

Reproduce el bootstrap del notebook 05.


,bloque_dias,n_bloques,comparador,delta,delta_inf,delta_sup,P(delta>=0),mejora,mejora_inf,mejora_sup
0,7,52,Estacional semanal,-0.0255,-0.0319,-0.0202,0.0,0.5060,0.4531,0.5483
4,7,52,Ridge,-0.0161,-0.0173,-0.0148,0.0,0.3917,0.3633,0.4198
8,7,52,Árbol podado,-0.0083,-0.0095,-0.0073,0.0,0.2502,0.2250,0.2758
12,7,52,RF dummies,-0.0012,-0.0018,-0.0008,0.0,0.0470,0.0306,0.0653


## 2. Sensibilidad al largo del bloque

Se repite la comparación macro de cada comparador con bloques de 1, 3, 7, 14 y 28 días (2.000 réplicas, semilla 42).
Si el intervalo cambiara mucho con el largo, la conclusión dependería de una elección arbitraria.

In [ ]:
CRONO.seccion('2. Sensibilidad al largo del bloque')

In [4]:
sens_blocks = pd.concat([run_bootstrap(L, comparators) for L in [1, 3, 7, 14, 28]], ignore_index=True)
sens_blocks = sens_blocks[sens_blocks.nivel == "Macro"].drop(columns=["nivel"])
sens_blocks.to_csv(RESULTS_DIR / "bootstrap_block_length.csv", index=False)
(sens_blocks.pivot(index="comparador", columns="bloque_dias", values="mejora").mul(100).round(1))

bloque_dias,1,3,7,14,28
comparador,,,,,
Estacional semanal,50.6,50.6,50.6,50.6,50.6
RF dummies,4.7,4.7,4.7,4.7,4.7
Ridge,39.2,39.2,39.2,39.2,39.2
Árbol podado,25.0,25.0,25.0,25.0,25.0


In [5]:
ancho = sens_blocks.assign(ancho_ic=100 * (sens_blocks.mejora_sup - sens_blocks.mejora_inf))
ancho.pivot(index="comparador", columns="bloque_dias", values="ancho_ic").round(1)

bloque_dias,1,3,7,14,28
comparador,,,,,
Estacional semanal,6.4,7.8,9.5,8.2,8.6
RF dummies,2.8,3.0,3.5,4.4,4.1
Ridge,5.5,6.4,5.6,6.3,7.1
Árbol podado,4.8,4.8,5.1,5.2,3.5


## Lectura

- El valor observado no depende del largo del bloque; solo cambian los intervalos.
- La última tabla muestra el ancho del IC 95 % de la mejora relativa (puntos porcentuales) según el largo del bloque. Si crece con bloques más largos, hay dependencia entre días cercanos que los bloques cortos no capturan.

## 3. Tablas de las búsquedas de hiperparámetros

El notebook 03 exporta estas tablas (`ridge_search.csv`, `tree_search.csv`, `rf_search.csv`) cuando se ejecuta la búsqueda completa (`python analisis.py --busqueda-completa`). Como esa búsqueda tarda varias horas, esta celda las reconstruye a partir de las salidas guardadas de ese notebook y las escribe a `results/`, de modo que el informe pueda generarse sin repetirla.

In [ ]:
CRONO.seccion('3. Tablas de las búsquedas de hiperparámetros')

In [6]:
import io, json, re

nb03 = json.load(open(ROOT / "notebooks" / "03_model_development.ipynb", encoding="utf8"))


def find_output(header_regex):
    for c in nb03["cells"]:
        for o in c.get("outputs", []):
            t = "".join(o.get("data", {}).get("text/plain", []))
            if re.match(header_regex, t.lstrip("\n")):
                return t
    raise KeyError(header_regex)


def parse_wrapped(t):
    blocks = [b for b in re.split(r"\n\s*\n", t.strip("\n")) if b.strip()]
    parts = []
    for b in blocks:
        lines = [l.rstrip().rstrip("\\") for l in b.splitlines()]
        parts.append(pd.read_csv(io.StringIO("\n".join(lines)), sep=r"\s+", index_col=0))
    return pd.concat(parts, axis=1)


ridge_search = parse_wrapped(find_output(r"\s*alpha\s+mean_NMAE"))
tree_search = parse_wrapped(find_output(r"\s*ccp_alpha\s+mean_NMAE"))
rf_search = parse_wrapped(find_output(r"\s*config_id\s+n_estimators"))

assert abs(ridge_search.iloc[0]["mean_NMAE"] - 0.04282) < 1e-5 and abs(rf_search.iloc[0]["mean_nmae"] - 0.02821) < 1e-5
assert len(ridge_search) == 15 and len(tree_search) == 20 and len(rf_search) == 18

ridge_search.to_csv(RESULTS_DIR / "ridge_search.csv", index=False)
tree_search.to_csv(RESULTS_DIR / "tree_search.csv", index=False)
rf_search.to_csv(RESULTS_DIR / "rf_search.csv", index=False)
rf_search.round(5)

,config_id,n_estimators,max_depth,min_samples_leaf,max_features,mean_nmae,std_nmae,min_nmae,max_nmae,n_windows
0,0,200,NaN,2,0.7,0.02821,0.00085,0.02724,0.02880,3
1,12,200,25.0,2,0.7,0.02822,0.00084,0.02726,0.02878,3
2,6,200,15.0,2,0.7,0.02847,0.00072,0.02765,0.02890,3
3,14,200,25.0,5,0.7,0.02853,0.00084,0.02756,0.02910,3
4,2,200,NaN,5,0.7,0.02856,0.00084,0.02760,0.02912,3
5,8,200,15.0,5,0.7,0.02869,0.00079,0.02778,0.02916,3
6,1,200,NaN,2,1.0,0.02894,0.00100,0.02779,0.02962,3
7,13,200,25.0,2,1.0,0.02894,0.00101,0.02779,0.02964,3
8,7,200,15.0,2,1.0,0.02918,0.00088,0.02816,0.02969,3
9,3,200,NaN,5,1.0,0.02919,0.00096,0.02809,0.02983,3


## 4. Tamaños de las ventanas de validación

Casos completos de entrenamiento y validación de cada ventana (suma de los tres países), exportados a `results/window_sizes.csv`.

In [ ]:
CRONO.seccion('4. Tamaños de las ventanas de validación')

In [7]:
from modeling import make_partitions

parts = make_partitions(df)
window_sizes = pd.DataFrame([{"window": w, "n_train": len(p["train"]), "n_val": len(p["validation"])}
                             for w, p in parts.items()])
window_sizes.to_csv(RESULTS_DIR / "window_sizes.csv", index=False)
window_sizes

,window,n_train,n_val
0,W1,25484,26352
1,W2,51836,26087
2,W3,77923,25685


## 5. Modelo final: complejidad e importancia de los predictores

Se reajustan el Random Forest cíclico y el árbol podado con el desarrollo completo (2015–2018), igual que en el notebook 05.
Como verificación, se compara el NMAE del Random Forest reajustado con el de las predicciones guardadas en `results/test_predictions.csv`.
Un reajuste con otra versión de librerías o de equipo puede dar predicciones levemente distintas (el bosque usa aleatoriedad), por lo que se exige coincidencia aproximada y se informa la diferencia.
La importancia es la basada en impureza, agrupada por predictor original (seno y coseno de una misma variable se suman).

In [ ]:
CRONO.seccion('5. Modelo final: complejidad e importancia de los predictores')

In [8]:
import time
from modeling import make_rf, make_tree

_, test_df = make_final_split(df)

rf, feats = make_rf("cyclic")
t0 = time.perf_counter()
rf.fit(final_train[feats], final_train[TARGET])
t_rf = time.perf_counter() - t0

# verificación contra las predicciones guardadas (notebook 05)
chk = test_df[["utc_timestamp", "country", TARGET]].copy()
chk["new"] = rf.predict(test_df[feats])
chk = chk.merge(pred[["utc_timestamp", "country", "pred_rf"]], on=["utc_timestamp", "country"])
assert len(chk) == len(pred)
mu_s = pd.Series(mu, index=COUNTRIES)
check = []
for c in COUNTRIES:
    s_ = chk[chk.country == c]
    check.append({"country": c,
                  "NMAE reajuste (este equipo)": (s_[TARGET] - s_["new"]).abs().mean() / mu_s[c],
                  "NMAE guardado (notebook 05)": (s_[TARGET] - s_["pred_rf"]).abs().mean() / mu_s[c]})
check = pd.DataFrame(check)
check.loc[len(check)] = ["Macro", check.iloc[:, 1].mean(), check.iloc[:, 2].mean()]
check["dif. relativa %"] = 100 * (check.iloc[:, 1] / check.iloc[:, 2] - 1)
check["max |dif. predicción| (MW)"] = [(chk.loc[chk.country == c, "new"] - chk.loc[chk.country == c, "pred_rf"]).abs().max()
                                       for c in COUNTRIES] + [(chk["new"] - chk["pred_rf"]).abs().max()]
check.to_csv(RESULTS_DIR / "final_refit_check.csv", index=False)
display(check.round(5))
exact = bool(np.allclose(chk["new"], chk["pred_rf"], rtol=1e-6))
print("Reajuste idéntico al guardado:", exact, "| tiempo de ajuste: %.1f s" % t_rf)
assert abs(check.iloc[3, 3]) < 2, "el reajuste difiere más de 2 % del guardado"

forest = rf.named_steps["model"]
names = rf.named_steps["preprocessor"].get_feature_names_out()


def group(n):
    g = re.sub(r"^(num|cat)__", "", n)
    if g.startswith("country_"):
        return "country"
    return re.sub(r"_(sin|cos)$", "", g)


imp = (pd.Series(forest.feature_importances_, index=[group(n) for n in names])
       .groupby(level=0).sum().sort_values(ascending=False).rename("importancia").reset_index()
       .rename(columns={"index": "predictor"}))
imp.to_csv(RESULTS_DIR / "rf_importance_final.csv", index=False)

depths = [t.tree_.max_depth for t in forest.estimators_]
nodes = [t.tree_.node_count for t in forest.estimators_]
leaves = [t.tree_.n_leaves for t in forest.estimators_]

tree, tfeats = make_tree()
t0 = time.perf_counter()
tree.fit(final_train[tfeats], final_train[TARGET])
t_tree = time.perf_counter() - t0
tm_ = tree.named_steps["model"]

complexity = pd.DataFrame([
    {"modelo": "Random Forest (cíclico)", "arboles": len(forest.estimators_), "profundidad_media": np.mean(depths),
     "profundidad_max": np.max(depths), "nodos_totales": int(np.sum(nodes)), "hojas_totales": int(np.sum(leaves)),
     "segundos_ajuste": t_rf},
    {"modelo": "Árbol podado", "arboles": 1, "profundidad_media": tm_.get_depth(), "profundidad_max": tm_.get_depth(),
     "nodos_totales": int(tm_.tree_.node_count), "hojas_totales": int(tm_.get_n_leaves()), "segundos_ajuste": t_tree}])
complexity.to_csv(RESULTS_DIR / "final_model_complexity.csv", index=False)
display(complexity.round(2))
imp.round(4)

,country,NMAE reajuste (este equipo),NMAE guardado (notebook 05),dif. relativa %,max |dif. predicción| (MW)
0,DE,0.02427,0.02424,0.10637,2200.49561
1,ES,0.02030,0.02041,-0.54095,774.23030
2,FR,0.03013,0.03014,-0.02678,1690.27798
3,Macro,0.02490,0.02493,-0.12392,2200.49561


Reajuste idéntico al guardado: False | tiempo de ajuste: 16.3 s


,modelo,arboles,profundidad_media,profundidad_max,nodos_totales,hojas_totales,segundos_ajuste
0,Random Forest (cíclico),200,35.18,44,11412818,5706509,16.32
1,Árbol podado,1,26.00,26,8521,4261,119.11


,predictor,importancia
0,lag_168h,0.6468
1,demand_mw,0.2474
2,lag_48h,0.0499
3,target_day,0.0161
4,lag_72h,0.0150
5,rolling_mean_168h,0.0089
6,rolling_mean_24h,0.0083
7,target_hour,0.0035
8,target_month,0.0029
9,country,0.0012


## 6. Correlación entre predictores

Correlación de Pearson entre los seis predictores numéricos y la respuesta en el entrenamiento final, calculada **dentro de cada país**
(las escalas de los países son muy distintas y mezclarlos inflaría las correlaciones). Se guarda por país y el promedio de los tres.

In [ ]:
CRONO.seccion('6. Correlación entre predictores')

In [9]:
num_cols = ["demand_mw", "lag_48h", "lag_72h", "lag_168h", "rolling_mean_24h", "rolling_mean_168h", TARGET]
mats = {c: final_train.loc[final_train.country == c, num_cols].corr() for c in COUNTRIES}
rows = []
for i, a in enumerate(num_cols):
    for b in num_cols[i + 1:]:
        vals = {c: mats[c].loc[a, b] for c in COUNTRIES}
        rows.append({"var_a": a, "var_b": b, **vals, "promedio": np.mean(list(vals.values()))})
corr = pd.DataFrame(rows)
corr.to_csv(RESULTS_DIR / "predictor_correlation.csv", index=False)
corr.round(3)

,var_a,var_b,DE,ES,FR,promedio
0,demand_mw,lag_48h,0.758,0.814,0.924,0.832
1,demand_mw,lag_72h,0.523,0.641,0.842,0.669
2,demand_mw,lag_168h,0.718,0.753,0.855,0.776
3,demand_mw,rolling_mean_24h,0.563,0.542,0.897,0.667
4,demand_mw,rolling_mean_168h,0.313,0.357,0.836,0.502
5,demand_mw,target_demand_mw,0.758,0.814,0.924,0.832
6,lag_48h,lag_72h,0.758,0.814,0.924,0.832
7,lag_48h,lag_168h,0.502,0.611,0.801,0.638
8,lag_48h,rolling_mean_24h,0.546,0.529,0.893,0.656
9,lag_48h,rolling_mean_168h,0.336,0.382,0.850,0.522


## 7. Datos ausentes: dónde caen y si sesgan la comparación entre países

**(a) Bloques de ausentes.** Horas consecutivas sin demanda observada, por país (`results/missing_blocks.csv`).

**(b) Evaluación en fechas comunes.** Cada país se evalúa con sus propios casos completos, de modo que Francia (con más pérdidas) se evalúa en
menos fechas que los demás. Para ver si eso distorsiona la comparación, se recalcula el NMAE de cada país restringido a las fechas en que
**los tres países** tienen caso completo. En desarrollo se usan los procedimientos que no requieren búsqueda (referencias y Ridge) en W1–W3;
en 2019 se usan también las predicciones guardadas del Random Forest cíclico.

In [ ]:
CRONO.seccion('7. Datos ausentes: dónde caen y si sesgan la comparación entre países')

In [10]:
# (a) bloques de horas ausentes en la demanda observada
miss = df.loc[df["demand_mw"].isna(), ["country", "utc_timestamp"]].sort_values(["country", "utc_timestamp"])
blocks = []
for c, g in miss.groupby("country"):
    t = g["utc_timestamp"].reset_index(drop=True)
    new = (t.diff() != pd.Timedelta(hours=1)).cumsum()
    for _, b in t.groupby(new):
        blocks.append({"country": c, "inicio_utc": b.iloc[0], "horas": len(b)})
blocks = pd.DataFrame(blocks)
blocks.to_csv(RESULTS_DIR / "missing_blocks.csv", index=False)
print(blocks.groupby("country").agg(bloques=("horas", "size"), horas=("horas", "sum")))
blocks

         bloques  horas
country                
ES             4     18
FR             6     30


,country,inicio_utc,horas
0,ES,2015-01-01 00:00:00+00:00,1
1,ES,2015-01-02 06:00:00+00:00,3
2,ES,2015-01-02 11:00:00+00:00,8
3,ES,2015-05-01 10:00:00+00:00,6
4,FR,2015-01-01 00:00:00+00:00,1
5,FR,2017-02-05 19:00:00+00:00,13
6,FR,2018-08-12 07:00:00+00:00,5
7,FR,2018-10-04 01:00:00+00:00,3
8,FR,2018-12-14 04:00:00+00:00,3
9,FR,2019-07-25 00:00:00+00:00,5


In [11]:
from modeling import evaluate_predictions, make_ridge

rows = []


for w, p in parts.items():
    tr, va = p["train"], p["validation"].copy()
    ridge_m, rf_feats = make_ridge("dummy")
    ridge_m.fit(tr[rf_feats], tr[TARGET])
    va["pred_Persistencia"] = va["demand_mw"]
    va["pred_Estacional semanal"] = va["lag_168h"]
    va["pred_Ridge"] = ridge_m.predict(va[rf_feats])
    ncount = va.groupby("utc_timestamp")["country"].nunique()
    common = va[va["utc_timestamp"].isin(ncount[ncount == 3].index)]
    for name in ["Persistencia", "Estacional semanal", "Ridge"]:
        own = evaluate_predictions(tr, va, va[f"pred_{name}"], name, w)
        com = evaluate_predictions(tr, common, common[f"pred_{name}"], name, w)
        for c in COUNTRIES + ["Macro"]:
            o, k = own[own.country == c].iloc[0], com[com.country == c].iloc[0]
            rows.append({"periodo": w, "modelo": name, "pais": c, "n_propio": o["n"], "n_comun": k["n"],
                         "nmae_propio": o["NMAE"], "nmae_comun": k["NMAE"]})

# 2019: predicciones guardadas; la escala es la media del entrenamiento final
t19 = pred.copy()
ncount = t19.groupby("utc_timestamp")["country"].nunique()
common19 = t19[t19["utc_timestamp"].isin(ncount[ncount == 3].index)]
mu_s = pd.Series(mu, index=COUNTRIES)
for name, col in [("Persistencia", "pred_persistence"), ("Estacional semanal", "pred_weekly"),
                  ("Ridge", "pred_ridge"), ("Árbol podado", "pred_tree"), ("RF cíclico", "pred_rf")]:
    nm_own, nm_com = {}, {}
    for c in COUNTRIES:
        for dset, store in [(t19, nm_own), (common19, nm_com)]:
            s = dset[dset.country == c]
            store[c] = ((s[TARGET] - s[col]).abs().mean() / mu_s[c], len(s))
    for c in COUNTRIES:
        rows.append({"periodo": "2019", "modelo": name, "pais": c, "n_propio": nm_own[c][1], "n_comun": nm_com[c][1],
                     "nmae_propio": nm_own[c][0], "nmae_comun": nm_com[c][0]})
    rows.append({"periodo": "2019", "modelo": name, "pais": "Macro",
                 "n_propio": sum(v[1] for v in nm_own.values()), "n_comun": sum(v[1] for v in nm_com.values()),
                 "nmae_propio": np.mean([v[0] for v in nm_own.values()]),
                 "nmae_comun": np.mean([v[0] for v in nm_com.values()])})

effect = pd.DataFrame(rows)
effect["cambio_pct"] = 100 * (effect["nmae_comun"] / effect["nmae_propio"] - 1)
effect.to_csv(RESULTS_DIR / "missing_effect.csv", index=False)
effect[effect.modelo.isin(["Estacional semanal", "Ridge", "RF cíclico"])].round(4)

,periodo,modelo,pais,n_propio,n_comun,nmae_propio,nmae_comun,cambio_pct
4,W1,Estacional semanal,DE,8784,8784,0.0403,0.0403,0.0000
5,W1,Estacional semanal,ES,8784,8784,0.0453,0.0453,0.0000
6,W1,Estacional semanal,FR,8784,8784,0.0648,0.0648,0.0000
7,W1,Estacional semanal,Macro,26352,26352,0.0501,0.0501,0.0000
8,W1,Ridge,DE,8784,8784,0.0409,0.0409,0.0000
9,W1,Ridge,ES,8784,8784,0.0407,0.0407,0.0000
10,W1,Ridge,FR,8784,8784,0.0434,0.0434,0.0000
11,W1,Ridge,Macro,26352,26352,0.0417,0.0417,0.0000
16,W2,Estacional semanal,DE,8760,8567,0.0443,0.0447,0.9254
17,W2,Estacional semanal,ES,8760,8567,0.0512,0.0518,1.2032


In [ ]:
# Tiempo por seccion de este notebook
CRONO.guardar()